# AorticTurb — canonical Pass 3 execution

This notebook is a **thin execution driver** for the public `khalid-saqr/AorticTurb` package. It does not reimplement turbulence calculations.

Canonical scientific scope for this run:

1. mandatory startup/integrity gate;
2. fresh phase-conditioned decomposition for M/AAo at `t018`, `t021`, `t024`, with 16 independent phase-locked realizations per phase;
3. Gate A using the unchanged primary implementation;
4. the pre-specified support-policy sensitivity analysis.

The notebook does **not** call `run-all`, KO, Taylor, Lumley, canonical diagnostics, integrated classification, or promote anything into root `05_results`.

Scientific package commit pinned for this run: `4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5`.


In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

AORTICTURB_COMMIT = "4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5"
AORTICTURB_SPEC = f"git+https://github.com/khalid-saqr/AorticTurb.git@{AORTICTURB_COMMIT}"

VENV_DIR = Path("/content/aorticturb-pass3-venv")
if VENV_DIR.exists():
    shutil.rmtree(VENV_DIR)

subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
VENV_PYTHON = VENV_DIR / "bin" / "python"
AORTICTURB = VENV_DIR / "bin" / "aorticturb"

subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--quiet", "--disable-pip-version-check", AORTICTURB_SPEC],
    check=True,
)

probe = subprocess.run(
    [
        str(VENV_PYTHON),
        "-c",
        (
            "import json;"
            "from importlib import metadata;"
            "d=metadata.distribution('aorticturb');"
            "u=json.loads(d.read_text('direct_url.json'));"
            "print(u['vcs_info']['commit_id'])"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
INSTALLED_COMMIT = probe.stdout.strip()
if INSTALLED_COMMIT != AORTICTURB_COMMIT:
    raise RuntimeError(
        f"Installed AorticTurb commit mismatch: expected {AORTICTURB_COMMIT}, got {INSTALLED_COMMIT}"
    )

version = subprocess.run(
    [str(AORTICTURB), "version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(version)
print("Pinned scientific commit:", INSTALLED_COMMIT)
print("Isolated interpreter:", VENV_PYTHON)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study").resolve()
DATASET_ROOT = PROJECT_ROOT
WORKSPACE_ROOT = PROJECT_ROOT / "results"
ANALYSIS_ROOT = WORKSPACE_ROOT / "04_analysis" / "01_laminar_vs_unsteady"

required = [
    PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "METADATA.json",
    PROJECT_ROOT / "SHA256SUMS.txt",
    PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / "t018",
    PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / "t021",
    PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / "t024",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError("Canonical project root is incomplete:\n" + "\n".join(missing))

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Analysis workspace:", WORKSPACE_ROOT)
print("Canonical root 05_results is not an output target:", PROJECT_ROOT / "05_results")


In [ ]:
def run_cli(*args):
    command = [str(AORTICTURB), *map(str, args)]
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"AorticTurb command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed

def runs_with(marker):
    if not ANALYSIS_ROOT.exists():
        return set()
    return {
        path.resolve()
        for path in ANALYSIS_ROOT.iterdir()
        if path.is_dir() and (path / marker).is_file()
    }

def run_stage(marker, *args):
    before = runs_with(marker)
    run_cli(*args)
    after = runs_with(marker)
    created = sorted(after - before)
    if len(created) != 1:
        raise RuntimeError(
            f"Expected exactly one new run containing {marker}; found {len(created)}: {created}"
        )
    return created[0]


## 1. Mandatory startup gate

This runs the package startup gate before any new scientific analysis. A failure stops the notebook.


In [ ]:
run_cli("gate", "--root", DATASET_ROOT)
print("Startup gate: PASS")


## 2. Fresh phase-conditioned decomposition

No phases are pooled. Source NaNs remain measurement-support masks; raw data are not modified.


In [ ]:
DECOMPOSITION_RUN = run_stage(
    "decomposition_summary.json",
    "decompose",
    "--root", DATASET_ROOT,
    "--output", WORKSPACE_ROOT,
)
print("Decomposition run:", DECOMPOSITION_RUN)


## 3. Gate A — turbulence versus deterministic unsteadiness

This uses the package's existing Gate-A implementation and its unchanged primary support rule.


In [ ]:
REGIME_RUN = run_stage(
    "regime_summary.json",
    "regime",
    "--root", DATASET_ROOT,
    "--decomposition", DECOMPOSITION_RUN,
    "--output", WORKSPACE_ROOT,
)
print("Gate-A run:", REGIME_RUN)


## 4. Pre-specified support-policy sensitivity analysis

This evaluates the locked support-policy comparators against the fresh decomposition and Gate-A evidence. It does not replace the primary Gate-A result.


In [ ]:
SENSITIVITY_RUN = run_stage(
    "sensitivity_summary.json",
    "support-sensitivity",
    "--root", DATASET_ROOT,
    "--decomposition", DECOMPOSITION_RUN,
    "--regime", REGIME_RUN,
    "--output", WORKSPACE_ROOT,
)
print("Support-sensitivity run:", SENSITIVITY_RUN)


## 5. Validate and display the fresh evidence

The checks below are execution/provenance checks, not new turbulence criteria.


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

PHASES = {"t018", "t021", "t024"}
POLICIES = {
    "legacy_global_fraction",
    "strict_mask_consistency",
    "fully_finite_local_windows",
    "strict_mask_and_local_windows",
}

decomposition_metrics = pd.read_csv(DECOMPOSITION_RUN / "metrics.csv")
required_support_columns = {
    "phase",
    "support_fraction",
    "common_support_nodes",
    "union_support_nodes",
    "intersection_over_union_fraction",
    "masks_identical",
    "realizations",
}
missing_columns = required_support_columns - set(decomposition_metrics.columns)
if missing_columns:
    raise RuntimeError(f"Fresh decomposition is missing required support columns: {sorted(missing_columns)}")
if set(decomposition_metrics["phase"]) != PHASES:
    raise RuntimeError("Fresh decomposition does not contain exactly t018, t021, t024")
if set(decomposition_metrics["realizations"].astype(int)) != {16}:
    raise RuntimeError("Fresh decomposition does not report 16 realizations per phase")

regime_summary = json.loads((REGIME_RUN / "regime_summary.json").read_text())
sensitivity = pd.read_csv(SENSITIVITY_RUN / "support_sensitivity.csv")
if set(sensitivity["policy"]) != POLICIES:
    raise RuntimeError(f"Unexpected support-policy set: {sorted(set(sensitivity['policy']))}")

for run_dir in (DECOMPOSITION_RUN, REGIME_RUN, SENSITIVITY_RUN):
    manifest = json.loads((run_dir / "run_manifest.json").read_text())
    if manifest.get("git_commit") != AORTICTURB_COMMIT:
        raise RuntimeError(
            f"Run manifest commit mismatch in {run_dir}: {manifest.get('git_commit')} != {AORTICTURB_COMMIT}"
        )
    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(f"Run did not complete cleanly: {run_dir}")

support_table = decomposition_metrics[
    [
        "phase",
        "support_fraction",
        "common_support_nodes",
        "union_support_nodes",
        "intersection_over_union_fraction",
        "masks_identical",
        "realizations",
    ]
].copy()

sensitivity_table = sensitivity[
    [
        "phase",
        "policy",
        "is_primary_policy",
        "support_evaluable",
        "support_ready",
        "projected_decision",
        "primary_decision",
        "common_intersection_fraction_total",
        "intersection_over_union_fraction",
        "fully_finite_window_count",
        "local_window_min_valid_fraction",
    ]
].copy()

display(support_table)
display(sensitivity_table)
print("Primary Gate A:", regime_summary["phases"])


In [ ]:
record_dir = WORKSPACE_ROOT / "colab_runs"
record_dir.mkdir(parents=True, exist_ok=True)
record_path = record_dir / (
    "pass3_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + ".json"
)

record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "aorticturb_commit": AORTICTURB_COMMIT,
    "dataset_root": str(DATASET_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "decomposition_run": str(DECOMPOSITION_RUN),
    "regime_run": str(REGIME_RUN),
    "support_sensitivity_run": str(SENSITIVITY_RUN),
    "primary_gate_a": regime_summary["phases"],
    "support_metrics": support_table.to_dict(orient="records"),
    "support_sensitivity": sensitivity_table.to_dict(orient="records"),
    "canonical_root_05_results_written_by_notebook": False,
}
record_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Pass-3 execution record:", record_path)
print("Canonical root 05_results written by notebook: False")
